# Laboratorio: reconstrucción de frontend desde capturas

Este laboratorio sigue la aplicación real FastAPI + Streamlit + MongoDB + proveedor Gemini u OpenAI. Para probar Gemini, configura `AI_PROVIDER=gemini`, `GEMINI_API_KEY` y `GEMINI_MODEL` en `.env`; el nivel gratuito depende del proyecto y su cuota. El prototipo original era un clasificador demostrativo con datos sintéticos; no generaba interfaces. La generación no ajusta los pesos de un modelo generativo; opcionalmente puedes entrenar un índice visual k-NN con WebSight en la sección 5.1. Una captura no contiene el código fuente original y el resultado puede diferir en fuentes, interacciones, diseño responsive y contenido ausente.


## 1. Objetivo, alcance y arquitectura

Streamlit carga captura o URL pública; FastAPI valida, llama al modelo de visión y guarda el HTML/CSS/JavaScript en MongoDB. La vista previa se aísla en un iframe sandbox y el servidor nunca ejecuta el código generado. URLs privadas, páginas con login, contenido dinámico y bloqueo de automatización tienen limitaciones.

## 2. Preparación del entorno Conda en macOS

Ejecuta en Anaconda Terminal desde la raíz del proyecto:

```bash
conda create -n screenshot-ui python=3.11 -y
conda activate screenshot-ui
python -m pip install -r requirements.txt
python -m playwright install chromium
```

In [ ]:
from pathlib import Path
import os, sys, requests, json
ROOT = Path.cwd().resolve()
if not (ROOT / 'backend' / 'api.py').exists():
    raise RuntimeError('Abre este notebook desde la raíz del proyecto.')
sys.path.insert(0, str(ROOT))
print('Raíz:', ROOT)

## 3. Configuración de `.env` sin mostrar secretos

Para Gemini configura `AI_PROVIDER=gemini`, `GEMINI_API_KEY` y `GEMINI_MODEL` (ejemplo `gemini-3.5-flash-lite`). Para OpenAI configura `AI_PROVIDER=openai` y `OPENAI_API_KEY`. En ambos casos configura `MONGO_URI`, `MONGO_DB` y `API_URL`. El nivel gratuito de Gemini puede tener límites de uso y Google puede utilizar datos enviados para mejorar sus productos; evita capturas sensibles. Nunca imprimas ni guardes claves en salidas del notebook.


In [ ]:
from dotenv import load_dotenv
load_dotenv(ROOT / '.env', override=True)
provider = os.getenv('AI_PROVIDER','openai').strip().lower()
key_name = 'GEMINI_API_KEY' if provider == 'gemini' else 'OPENAI_API_KEY'
status = {'AI_PROVIDER': provider, key_name: bool(os.getenv(key_name)), 'MONGO_URI': bool(os.getenv('MONGO_URI')), 'MONGO_DB': bool(os.getenv('MONGO_DB')), 'API_URL': bool(os.getenv('API_URL'))}
print('Configuración detectada (sin valores secretos):', status)
if not all(value for key,value in status.items() if key != 'AI_PROVIDER'): print('Falta configuración: edita .env y vuelve a ejecutar esta celda.')

## 4. Verificación de dependencias y conexión a MongoDB

MongoDB crea la base y colección tras la primera inserción. En Compass conecta al URI local configurado y revisa `web_screenshot_builder.projects`.

In [ ]:
from importlib.util import find_spec
for module in ['fastapi','streamlit','pymongo','PIL','openai','playwright']:
    print(module, 'OK' if find_spec(module) else 'FALTA')
try:
    from pymongo import MongoClient
    client = MongoClient(os.getenv('MONGO_URI','mongodb://127.0.0.1:27017'), serverSelectionTimeoutMS=2500)
    client.admin.command('ping')
    print('MongoDB conectado; base:', os.getenv('MONGO_DB','web_screenshot_builder'))
except Exception as exc:
    print('MongoDB no disponible. Inicia MongoDB y verifica MONGO_URI en .env. (' + type(exc).__name__ + ')')

## 5.1. (Opcional) Entrenar el índice de referencias WebSight

La aplicación funciona aunque no ejecutes este paso. La muestra de WebSight permite recuperar hasta dos pares visualmente similares como contexto adicional; no entrena los pesos de Gemini ni de OpenAI. La primera descarga puede transferir un shard remoto grande. WebSight usa licencia CC BY 4.0: conserva la atribución si distribuyes el modelo o la aplicación.

Cambia `ENTRENAR_INDICE` a `True` y ejecuta esta celda solo si quieres descargar la muestra y entrenar el índice local.


In [ ]:
ENTRENAR_INDICE = False
if ENTRENAR_INDICE:
    from descargar_dataset import download_dataset
    from backend.ml import train
    resultado = download_dataset(max_samples=128, output_dir=ROOT / 'data' / 'processed')
    modelo = train(ROOT / 'data' / 'processed', ROOT / 'models' / 'modelo.joblib')
    print('Muestras descargadas:', resultado['samples'])
    print('Pares indexados:', len(modelo['examples']))
else:
    print('Índice sin cambios. Activa ENTRENAR_INDICE solo si quieres descargar WebSight.')


## 5. Carga de captura o captura desde URL

Selecciona el archivo con el botón o escribe una URL pública en el campo. No es necesario responder preguntas en la consola. La API captura URLs con Playwright Chromium; páginas que requieren sesión, con bloqueo o contenido dinámico pueden fallar o verse incompletas. Después de seleccionar el origen, ejecuta la celda siguiente.


In [ ]:
from IPython.display import display, Image as DisplayImage
import ipywidgets as widgets
from pathlib import Path
source_selector = widgets.ToggleButtons(options=[('Subir imagen','archivo'),('Usar URL','url')], description='Origen:')
image_upload = widgets.FileUpload(accept='.jpg,.jpeg,.png,.webp', multiple=False, description='Elegir imagen')
url_input = widgets.Text(value='', placeholder='https://example.com', description='URL pública:', layout=widgets.Layout(width='80%'))
display(source_selector, image_upload, url_input)
image_path = None
source_url = None
image_bytes = None
print('Elige una imagen o ingresa una URL. Luego ejecuta la siguiente celda de validación.')


## 6. Validación y visualización de la imagen

El backend valida contenido real, tipo, tamaño máximo de 10 MB y 25 megapíxeles. Esta celda valida la carga local; en URL la API aplica la misma validación sobre la captura.

In [ ]:
image_bytes = None
image_path = None
source_url = None
if source_selector.value == 'archivo':
    files = image_upload.value
    if files:
        item = next(iter(files.values())) if isinstance(files, dict) else files[0]
        filename = item.get('name', 'captura')
        image_bytes = bytes(item['content'])
        image_path = Path(filename)
        try:
            image = Image.open(__import__('io').BytesIO(image_bytes)); image.verify()
            image = Image.open(__import__('io').BytesIO(image_bytes))
            if image.format not in ('PNG','JPEG','WEBP') or len(image_bytes)>10*1024*1024 or image.width*image.height>25_000_000: raise ValueError('Formato o tamaño fuera de límites')
            print('Imagen válida:', image.format, image.size, f'{len(image_bytes)/1024/1024:.2f} MB')
            display(DisplayImage(data=image_bytes))
        except Exception as exc:
            image_bytes = None; print('Imagen no válida:', type(exc).__name__, str(exc))
    else: print('Selecciona una imagen en el control de carga de la celda anterior.')
else:
    source_url = url_input.value.strip()
    if source_url: print('URL seleccionada. La API comprobará el dominio y capturará la página.')
    else: print('Escribe una URL pública en el campo de la celda anterior.')


## 7. Llamada al backend y generación del frontend

Inicia API (`uvicorn backend.api:app --reload --port 8000`) y MongoDB en terminales separadas. Se llama al backend real; no se muestra contenido de ejemplo como si fuera una generación. Requiere clave de proveedor válida.

In [ ]:
API = os.getenv('API_URL','http://127.0.0.1:8000').rstrip('/')
project = None
try:
    if image_bytes:
        name = image_path.name
        response = requests.post(API+'/projects', files={'file':(name,image_bytes,'image/png' if name.lower().endswith('.png') else 'image/jpeg')}, timeout=120)
    elif source_url:
        response = requests.post(API+'/projects/from-url', json={'url':source_url}, timeout=150)
    else: response = None; print('Falta una captura válida.')
    if response is not None:
        if response.ok: project=response.json(); print('Proyecto guardado:', project['id'], project['provider'], project['model'])
        else: print('Backend:', response.status_code, response.json().get('detail','Error sin detalle'))
except requests.RequestException as exc: print('No se pudo conectar al backend. Inícialo y verifica API_URL. ('+type(exc).__name__+')')

## 8. Visualización y descarga del resultado

Descarga ZIP con HTML/CSS/JavaScript. No se ejecuta código generado dentro del notebook.

In [ ]:
from IPython.display import display, HTML
if project:
    display(HTML('<h3>HTML</h3><pre>'+project['html'].replace('&','&amp;').replace('<','&lt;')+'</pre>'))
    print('CSS:\n', project['css'][:12000])
    print('JavaScript:\n', project.get('javascript','')[:12000])
    import zipfile
    output = ROOT / 'reconstruccion_frontend.zip'
    with zipfile.ZipFile(output,'w',zipfile.ZIP_DEFLATED) as z:
        z.writestr('index.html','<!doctype html><html><head><meta charset="utf-8"><link rel="stylesheet" href="styles.css"></head><body>'+project['html']+'<script src="script.js"></script></body></html>')
        z.writestr('styles.css',project['css']); z.writestr('script.js',project.get('javascript',''))
    print('Descarga lista:',output.name)
else: print('No hay resultado: revisa imagen, clave, API y MongoDB.')

## 9. Consulta y eliminación de proyectos guardados

Consulta desde la API; borrar requiere confirmar explícitamente el ID. También puedes gestionar proyectos en Streamlit o Compass.

In [ ]:
saved=[]
try:
    r=requests.get(API+'/projects',timeout=10); r.raise_for_status(); saved=r.json(); print(json.dumps(saved,ensure_ascii=False,indent=2))
except Exception as exc: print('No se pudo consultar: inicia API/MongoDB. ('+type(exc).__name__+')')
if saved:
    delete_id=input('Para borrar un proyecto, pega su ID; Enter conserva todos: ').strip()
    if delete_id:
        try:
            r=requests.delete(API+'/projects/'+delete_id,timeout=10); r.raise_for_status(); print('Eliminado:',delete_id)
        except Exception as exc: print('No se pudo eliminar:',type(exc).__name__)

## 10. Evaluación de similitud visual y limitaciones

Compara la captura original con la vista previa aislada en la aplicación. Evalúa estructura, color, espaciado, tipografía y respuesta a distintos anchos. Una métrica automática de píxeles puede penalizar antialiasing y variaciones de fuentes; una puntuación no implica fidelidad semántica. URL dinámica/login/bloqueos, contenido no visible, fuentes propietarias e interacciones originales limitan la reconstrucción. No existe código fuente original recuperable desde la imagen.

## 11. Pruebas y conclusiones

Desde Anaconda Terminal: `python -m pytest -q`. Las pruebas cubren validación y persistencia simulada sin llamadas pagadas ni MongoDB. El proveedor real requiere credencial propia y acceso al modelo configurado. La aplicación puede usar un índice opcional de referencias, pero no ajusta modelos generativos; la salida es código generado que solo se previsualiza en iframe sandbox.

In [ ]:
%pip install ipywidgets